In [2]:
# imports here
import pandas as pd
import numpy as np
from keras import Sequential
from keras.layers import Dense, Input
import keras as kr

In [3]:
# setting random seed so the result is same for every try
kr.utils.set_random_seed(42)

In [4]:
data_url="https://raw.githubusercontent.com/ziafaq/genai/refs/heads/main/diabetes.csv"
patient_df = pd.read_csv(data_url, header=None)

In [5]:
patient_df.describe()

,0,1,2,3,4,5,6,7,8
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [6]:
from keras import Sequential
from keras.optimizers import Adam
# Create model in a function
def create_model(input_dim:int, act_function:str, act_op_function:str):
  #model definition - a sequenctial model is created here
  model = Sequential()
  # add the input layer with the number of input columns
  model.add(Input(shape=(input_dim,)))
  # add the hidden layer
  model.add(Dense(2, activation=act_function))
  # add the output layer
  # by default the activation function will be linear
  model.add(Dense(1, activation=act_op_function))

  # Remove the compilation step here, KerasClassifier will handle it.
  # model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
  return model

In [7]:
#spliting the data for train and test
X=patient_df.iloc[:,:-1]
y=patient_df.iloc[:,-1]
X, y

(      0    1   2   3    4     5      6   7
 0     6  148  72  35    0  33.6  0.627  50
 1     1   85  66  29    0  26.6  0.351  31
 2     8  183  64   0    0  23.3  0.672  32
 3     1   89  66  23   94  28.1  0.167  21
 4     0  137  40  35  168  43.1  2.288  33
 ..   ..  ...  ..  ..  ...   ...    ...  ..
 763  10  101  76  48  180  32.9  0.171  63
 764   2  122  70  27    0  36.8  0.340  27
 765   5  121  72  23  112  26.2  0.245  30
 766   1  126  60   0    0  30.1  0.349  47
 767   1   93  70  31    0  30.4  0.315  23
 
 [768 rows x 8 columns],
 0      1
 1      0
 2      1
 3      0
 4      1
       ..
 763    0
 764    0
 765    0
 766    1
 767    0
 Name: 8, Length: 768, dtype: int64)

In [12]:
y.unique()

array([1, 0])

In [1]:
!pip uninstall -y scikit-learn
!pip install scikit-learn==1.4.2
!pip install scikeras

Found existing installation: scikit-learn 1.4.2
Uninstalling scikit-learn-1.4.2:
  Successfully uninstalled scikit-learn-1.4.2
  Using cached scikit_learn-1.4.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached scikit_learn-1.4.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.2 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.


In [8]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
estimator = KerasClassifier(build_fn=lambda: create_model(input_dim=X.shape[1], act_function='relu', act_op_function='sigmoid'), verbose=0)
model_test= KerasClassifier(build_fn=lambda: create_model(input_dim=X.shape[1], act_function='relu', act_op_function='sigmoid'))

In [13]:
# define the grid search parameters
batch_size = [10, 20, 40]
epochs = [10, 50, 100]

# define the model parameters (passed to the build_fn)
model__act_function = ['relu', 'tanh']
model__act_op_function = ['sigmoid']
model__input_dim = [X.shape[1]]
optimizer = ['Adam', 'RMSprop']

param_grid = dict(batch_size=batch_size, epochs=epochs, model__act_function=model__act_function, model__act_op_function=model__act_op_function, model__input_dim=model__input_dim,
                  optimizer=optimizer)


# Create the KerasClassifier instance (estimator)
# Pass `create_model` directly as build_fn, and let GridSearchCV handle model__ prefixed parameters
# KerasClassifier also directly takes 'loss' and 'metrics' for compilation.
estimator = KerasClassifier(build_fn=create_model, loss='binary_crossentropy', metrics=['accuracy'], verbose=0)

# Create GridSearchCV
grid = GridSearchCV(estimator=estimator, param_grid=param_grid, n_jobs=1, cv=3)

In [14]:
# Fit the grid search to the data
grid_result = grid.fit(X, y)

/usr/local/lib/python3.12/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.12/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.12/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.12/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.12/dist-packages/scikeras

In [15]:
print("Best parameters found: %s" % grid_result.best_params_)

Best parameters found: {'batch_size': 40, 'epochs': 100, 'model__act_function': 'relu', 'model__act_op_function': 'sigmoid', 'model__input_dim': 8, 'optimizer': 'RMSprop'}


In [18]:
grid_result.best_params_.keys()

dict_keys(['batch_size', 'epochs', 'model__act_function', 'model__act_op_function', 'model__input_dim', 'optimizer'])

In [40]:
resultdf = pd.DataFrame(columns=list(grid_result.best_params_.keys()) + ['best_score'])

In [43]:
resultdf

,batch_size,epochs,model__act_function,model__act_op_function,model__input_dim,optimizer,best_score
0,40,100,relu,sigmoid,8,RMSprop,0.670573


In [42]:
display(resultdf)

,batch_size,epochs,model__act_function,model__act_op_function,model__input_dim,optimizer,best_score
0,40,100,relu,sigmoid,8,RMSprop,0.670573


Summary:
Certainly! Here is  a summary of the steps we have taken:

Initial Setup & Data Loading: We started by importing necessary libraries like pandas, numpy, and keras. We set a random seed for reproducibility and loaded the diabetes dataset into a pandas DataFrame (patient_df).

Data Splitting: The dataset was then split into features (X) and the target variable (y).

Keras Model Definition: A function create_model was defined to build a simple Keras Sequential neural network with an input layer, a hidden dense layer, and an output dense layer.

KerasClassifier and GridSearchCV Setup: We initialized a KerasClassifier (a scikit-learn wrapper for Keras models from scikeras) and GridSearchCV for hyperparameter tuning. We defined a parameter grid (param_grid) including batch_size, epochs, activation functions, input dimensions, and optimizers.

Error Resolution during Setup:

We encountered and resolved several issues during the setup:
A TypeError due to a typo ('optimer' instead of 'optimizer') in the create_model compile method.
Multiple AttributeErrors and TypeErrors related to KerasClassifier expecting create_model as a
function reference (not an already-called function) and create_model prematurely compiling the Keras model.

We removed the compilation step from create_model and passed loss and metrics directly to KerasClassifier.
Issues with GridSearchCV n_jobs=-1 (multiprocessing) that were resolved by setting n_jobs=1.

A ValueError where a list of activation functions was incorrectly passed to create_model during KerasClassifier initialization, instead of letting GridSearchCV pass individual strings.

Grid Search Execution:

After resolving the setup issues, grid.fit(X, y) was successfully executed to perform the hyperparameter search.

Result Extraction and Display:

Finally, we extracted the grid_result.best_params_ and grid_result.best_score_. We then created a pandas DataFrame (resultdf) to clearly display the best hyperparameters found by the grid search along with their corresponding performance score.

The best parameters found were a batch_size of 40, epochs of 100, relu activation for the hidden layer, sigmoid activation for the output layer, an input_dim of 8, and the RMSprop optimizer, yielding a best_score of approximately 0.6706.

In [46]:
# Splitting the dataset into test train split
#from sklearn.preprocessing import StandardScaler
#from sklearn.model_selection import train_test_split

#X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

from keras import Sequential
from keras.optimizers import Adam
# Create model in a function
def create_model_split(input_dim:int, act_function:str, act_op_function:str, no_of_neurons:int):
  #model definition - a sequenctial model is created here
  model = Sequential()
  # add the input layer with the number of input columns
  model.add(Input(shape=(input_dim,)))
  # add the hidden layer
  model.add(Dense(no_of_neurons, activation=act_function))
  # add the output layer
  # by default the activation function will be linear
  model.add(Dense(1, activation=act_op_function))

  # Remove the compilation step here, KerasClassifier will handle it.
  # model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
  return model

# define the grid search parameters
batch_size = [10, 20, 40]
epochs = [10, 50, 100]

# define the model parameters (passed to the build_fn)
model__act_function = ['relu', 'tanh']
model__act_op_function = ['sigmoid']
model__input_dim = [X.shape[1]]
model__no_of_neurons = [8]
optimizer = ['SGD', 'Adam', 'RMSprop']

param_grid_01 = dict(batch_size=batch_size, epochs=epochs, model__act_function=model__act_function, model__act_op_function=model__act_op_function, model__input_dim=model__input_dim,
                  optimizer=optimizer, model__no_of_neurons=model__no_of_neurons)


# Create the KerasClassifier instance (estimator)
# Pass `create_model` directly as build_fn, and let GridSearchCV handle model__ prefixed parameters
# KerasClassifier also directly takes 'loss' and 'metrics' for compilation.
estimator_01 = KerasClassifier(build_fn=create_model_split, loss='binary_crossentropy', metrics=['accuracy'], verbose=0, validation_split=0.2)

# Create GridSearchCV
grid_01 = GridSearchCV(estimator=estimator_01 = KerasClassifier(build_fn=create_model_split, loss='binary_crossentropy', metrics=['accuracy'], verbose=0, validation_split=0.2)
, param_grid=param_grid_01, n_jobs=1, cv=3)

# Fit the grid search to the data
grid_result_01 = grid_01.fit(X, y)

/usr/local/lib/python3.12/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.12/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.12/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.12/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.12/dist-packages/scikeras

In [48]:
records = []
for i in range(len(grid_result_01.cv_results_['params'])):
    params_dict = grid_result_01.cv_results_['params'][i]
    mean_score = grid_result_01.cv_results_['mean_test_score'][i]

    row_data = params_dict.copy()
    row_data['best_score'] = mean_score
    records.append(row_data)

resultdf_new = pd.DataFrame(records)

In [49]:
resultdf_new

,batch_size,epochs,model__act_function,model__act_op_function,model__input_dim,model__no_of_neurons,optimizer,best_score
0,10,10,relu,sigmoid,8,2,Adam,0.567708
1,10,10,relu,sigmoid,8,2,RMSprop,0.643229
2,10,10,relu,sigmoid,8,4,Adam,0.532552
3,10,10,relu,sigmoid,8,4,RMSprop,0.569010
4,10,10,relu,sigmoid,8,8,Adam,0.583333
...,...,...,...,...,...,...,...,...
103,40,100,tanh,sigmoid,8,2,RMSprop,0.589844
104,40,100,tanh,sigmoid,8,4,Adam,0.645833
105,40,100,tanh,sigmoid,8,4,RMSprop,0.640625
106,40,100,tanh,sigmoid,8,8,Adam,0.639323


In [50]:
grid_result_01.best_score_, grid_result_01.best_params_

(np.float64(0.7122395833333334),
 {'batch_size': 10,
  'epochs': 100,
  'model__act_function': 'relu',
  'model__act_op_function': 'sigmoid',
  'model__input_dim': 8,
  'model__no_of_neurons': 4,
  'optimizer': 'RMSprop'})

In [53]:
grid_result_01.predict(X)

ValueError: No axis named 2 for object type DataFrame

In [60]:
prediction = grid_result_01.predict(X.iloc[0:1])
print(f"Predicted output for the second row of X: {prediction}")

Predicted output for the second row of X: [1]


In [61]:
patient_df.iloc[0:1]

,0,1,2,3,4,5,6,7,8
0,6,148,72,35,0,33.6,0.627,50,1


In [59]:
patient_df[patient_df[8] == 1]

,0,1,2,3,4,5,6,7,8
0,6,148,72,35,0,33.6,0.627,50,1
2,8,183,64,0,0,23.3,0.672,32,1
4,0,137,40,35,168,43.1,2.288,33,1
6,3,78,50,32,88,31.0,0.248,26,1
8,2,197,70,45,543,30.5,0.158,53,1
...,...,...,...,...,...,...,...,...,...
755,1,128,88,39,110,36.5,1.057,37,1
757,0,123,72,0,0,36.3,0.258,52,1
759,6,190,92,0,0,35.5,0.278,66,1
761,9,170,74,31,0,44.0,0.403,43,1
